In [ ]:
import bisect
import csv
import concurrent.futures
import datetime
import faiss
import glob
import gc
import importlib.util
import json
import joblib
import math
import multiprocessing
import os
import pickle
import random
import re
import sys
import string
import subprocess
import time
import torch
import uuid
import yaml

import numpy as np
import pandas as pd
import seaborn as sns
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import multiprocessing as mp

from unsloth import FastLanguageModel

from accelerate import Accelerator, PartialState
from collections import Counter, defaultdict
from concurrent.futures import ProcessPoolExecutor, as_completed
from datasets import Dataset, concatenate_datasets, load_from_disk, IterableDataset, load_dataset
from datetime import timedelta
from imblearn.under_sampling import TomekLinks, NearMiss
from functools import partial
from math import inf
from multiprocessing import cpu_count, Pool
from pathlib import Path
from pandas import Timedelta
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType, PeftModel
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import precision_score, recall_score
from scipy import stats
from scipy.spatial import KDTree
from sentence_transformers import SentenceTransformer
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
    TrainingArguments, DataCollatorForLanguageModeling
)
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
from trl import SFTConfig, SFTTrainer, DPOTrainer, DPOConfig
from typing import List, Dict, Optional, Tuple, Union, Set

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
# =========================
# Repository-level configuration
# =========================

PROJECT_ROOT = Path.cwd()

INPUT_ROOT = PROJECT_ROOT / "input"
OUTPUT_ROOT = PROJECT_ROOT / "output"

# Change this value for another disk model, e.g., "ST12000NM0008" or "model-one".
model_serial_id = "ST12000NM0008"

DATA_DIR = OUTPUT_ROOT / "data" / model_serial_id
MONTHLY_TEXTUAL_DIR = DATA_DIR / "monthly_textual"
MONTHLY_TEXTUAL_PATTERN = "backblaze_textual_*.csv"
# MONTHLY_TEXTUAL_PATTERN = "alibaba_disk_textual_*.csv"

MODEL_DIR = OUTPUT_ROOT / "models" / model_serial_id
FINAL_ADAPTER_DIR = MODEL_DIR / "final_adapter"
CHECKPOINT_DIR = MODEL_DIR / "checkpoints"

INFERENCE_DATA_DIR = DATA_DIR / "inference_data"
INFERENCE_DATASETS_DIR = INFERENCE_DATA_DIR / "inference_datasets"
SAMPLED_INFERENCE_DATASETS_DIR = INFERENCE_DATA_DIR / "sampled_inference_datasets"
DF_TEST_PICKLE_PATH = INFERENCE_DATA_DIR / "df_test.pkl"

SMART_ATTRIBUTES_PATH = INPUT_ROOT / "smart_attributes.csv"
NUMERICAL_TEST_CSV_PATH = DATA_DIR / "test.csv"

ML_BASELINE_RESULTS_DIR = OUTPUT_ROOT / "results" / "ml_baselines"
RESULTS_DIR = OUTPUT_ROOT / "results"
# RESULT_PATH = RESULTS_DIR / f"{model_serial_id}.csv"

INFERENCE_DATA_DIR.mkdir(parents=True, exist_ok=True)
INFERENCE_DATASETS_DIR.mkdir(parents=True, exist_ok=True)
SAMPLED_INFERENCE_DATASETS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_DIR.mkdir(parents=True, exist_ok=True)
FINAL_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

ML_BASELINE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# option 0: full; 1: no rag; 2: no ml; 3: no rag or ml.
ABLATION_OPTION = 0

## Load Datasets

In [ ]:
def process_single_file(csv_file, seed=42):
    """
    Worker function: load, clean and lightly filter one CSV file.
    Returns the cleaned Dataset or raises exception.
    """
    try:
        raw_ds = load_dataset("csv", data_files=[csv_file])["train"]

        def clean_example(example):
            # Clean failure
            failure = example["failure"]
            if failure is None or (isinstance(failure, float) and pd.isna(failure)):
                failure = 0
            else:
                try:
                    failure = int(float(failure))
                except:
                    failure = 0
            example["failure"] = failure

            try:
                parsed = pd.to_datetime(example["date"])
                example["date"] = str(parsed.date())
            except:
                example["date"] = None

            return example

        cleaned_ds = raw_ds.map(clean_example, num_proc=3)  # small num_proc per file
        cleaned_ds = cleaned_ds.filter(lambda x: x["date"] is not None)

        return {
            "file": os.path.basename(csv_file),
            "dataset": cleaned_ds,
            "success": True
        }
    except Exception as e:
        return {
            "file": os.path.basename(csv_file),
            "error": str(e),
            "success": False
        }

In [ ]:
def load_historical_datasets(
    data_dir,
    pattern="*.csv",
    seed=42,
    max_workers=None,
    num_historical_months=15,
    healthy_sample_ratio=0.10,
):
    random.seed(seed)

    csv_files = glob.glob(os.path.join(data_dir, pattern))
    csv_files.sort(key=lambda f: os.path.basename(f))  # chronological order

    if not csv_files:
        raise ValueError(f"No files found matching pattern '{pattern}' in {data_dir}")

    csv_files = csv_files[:num_historical_months]

    print(f"Found {len(csv_files)} historical monthly files: {[os.path.basename(f) for f in csv_files]}")

    if max_workers is None:
        max_workers = min(8, len(csv_files))

    monthly_datasets = []
    errors = []

    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        future_to_file = {
            executor.submit(process_single_file, f, seed): f
            for f in csv_files
        }

        for future in tqdm(as_completed(future_to_file), total=len(csv_files), desc="Processing monthly files"):
            result = future.result()
            if result["success"]:
                monthly_datasets.append(result["dataset"])
            else:
                errors.append((result["file"], result["error"]))

    if errors:
        print("\nThe following files failed to process:")
        for fname, err in errors:
            print(f"  {fname} -> {err}")
        if not monthly_datasets:
            raise RuntimeError("All monthly files failed to process")

    full_ds = concatenate_datasets(monthly_datasets)
    print(f"Loaded historical data: {len(full_ds)} entries")

    all_serials = set(full_ds.unique("serial_number"))
    print(f"Total distinct serial_numbers across historical months: {len(all_serials)}")

    failing_examples = full_ds.filter(lambda x: x["failure"] == 1)
    failing_serials = set(failing_examples.unique("serial_number")) if len(failing_examples) > 0 else set()
    print(f"Failing serial_numbers: {len(failing_serials)}")

    healthy_serials = sorted(all_serials - failing_serials)
    print(f"Healthy serial_numbers: {len(healthy_serials)}")

    if not healthy_serials:
        print("No healthy disks found; returning only failing disks.")
        historical_ds = full_ds.filter(lambda x: x["serial_number"] in failing_serials)
        historical_ds = historical_ds.sort(["serial_number", "date"])
        return historical_ds

    # Randomly sample healthy disks without using description-length heuristics.
    num_healthy_to_sample = int(len(healthy_serials) * healthy_sample_ratio)

    if num_healthy_to_sample == 0 and len(healthy_serials) > 0:
        num_healthy_to_sample = 1

    rng = random.Random(seed)
    sampled_healthy_serials = rng.sample(
        healthy_serials,
        k=min(num_healthy_to_sample, len(healthy_serials)),
    )

    print(
        f"Randomly sampled {len(sampled_healthy_serials)} healthy serials "
        f"from {len(healthy_serials)} healthy serials "
        f"(ratio={healthy_sample_ratio})."
    )

    selected_serials = failing_serials.union(set(sampled_healthy_serials))

    historical_ds = full_ds.filter(lambda x: x["serial_number"] in selected_serials)
    historical_ds = historical_ds.sort(["serial_number", "date"])

    total_entries = len(historical_ds)
    pos_entries = len(historical_ds.filter(lambda x: x["failure"] == 1))

    print("\n=== Historical RAG Dataset Summary ===")
    print(f"Total distinct serial_numbers: {len(selected_serials)}")
    print(f"   - Failing disks   : {len(failing_serials)}")
    print(f"   - Sampled healthy : {len(sampled_healthy_serials)}")
    print(f"Total entries: {total_entries}")
    print(f"   - Positive rows (failure==1): {pos_entries}")
    print(f"   - Negative rows             : {total_entries - pos_entries}")

    return historical_ds

In [ ]:
def load_csv_datasets(
    data_dir,
    pattern="*.csv",
    seed=42,
    max_workers=None,
    num_test_months=None,
    skip_first_n_months=15,
    healthy_sample_divisor=15,
):
    """
    Loads monthly CSV files, keeps all failing serials, and randomly samples
    a subset of healthy serials without any hand-crafted description-based
    filtering rules.

    - skip_first_n_months: number of earliest monthly files to skip before
      constructing the inference/test dataset.
    - num_test_months: optional number of monthly files to use after skipping.
    """

    random.seed(seed)
    np.random.seed(seed)

    csv_files = glob.glob(os.path.join(data_dir, pattern))
    csv_files.sort(key=lambda f: os.path.basename(f))  # chronological order

    if skip_first_n_months is not None and skip_first_n_months > 0:
        csv_files = csv_files[skip_first_n_months:]

    if num_test_months is not None:
        csv_files = csv_files[:num_test_months]

    if not csv_files:
        raise ValueError(f"No files found matching pattern '{pattern}' in {data_dir}")

    print(f"Found {len(csv_files)} monthly files: {[os.path.basename(f) for f in csv_files]}")

    if max_workers is None:
        max_workers = min(8, len(csv_files))

    monthly_datasets = []
    errors = []

    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        future_to_file = {
            executor.submit(process_single_file, f, seed): f
            for f in csv_files
        }

        for future in tqdm(
            as_completed(future_to_file),
            total=len(future_to_file),
            desc="Processing monthly files",
        ):
            result = future.result()
            if result["success"]:
                monthly_datasets.append(result["dataset"])
            else:
                errors.append((result["file"], result["error"]))

    if errors:
        print("\nThe following files failed to process:")
        for fname, err in errors:
            print(f"  {fname} -> {err}")
        if not monthly_datasets:
            raise RuntimeError("All monthly files failed to process")

    full_ds = concatenate_datasets(monthly_datasets)
    print(f"Loaded full raw test data: {len(full_ds)} entries")

    all_serials = set(full_ds.unique("serial_number"))
    print(f"Total distinct serial_numbers across all months: {len(all_serials)}")

    failing_examples = full_ds.filter(lambda x: x["failure"] == 1)
    failing_serials = set(failing_examples.unique("serial_number")) if len(failing_examples) > 0 else set()
    print(f"Failing serial_numbers (>=1 failure): {len(failing_serials)}")

    healthy_serials = sorted(all_serials - failing_serials)
    print(f"Healthy serial_numbers (no failures): {len(healthy_serials)}")

    num_healthy_to_sample = len(healthy_serials) // healthy_sample_divisor
    if num_healthy_to_sample == 0 and len(healthy_serials) > 0:
        num_healthy_to_sample = 1
        print("Warning: Very few healthy disks; sampling at least 1")

    rng = random.Random(seed)
    sampled_healthy_serials = rng.sample(
        healthy_serials,
        k=min(num_healthy_to_sample, len(healthy_serials)),
    )

    selected_serials = failing_serials.union(set(sampled_healthy_serials))

    test_ds = full_ds.filter(lambda x: x["serial_number"] in selected_serials)
    test_ds = test_ds.sort(["serial_number", "date"])

    total_entries = len(test_ds)
    pos_entries = len(test_ds.filter(lambda x: x["failure"] == 1))

    print("\n=== Final Test Dataset Summary ===")
    print(f"Total distinct serial_numbers: {len(selected_serials)}")
    print(f"   - Failing disks    : {len(failing_serials)}")
    print(f"   - Sampled healthy  : {len(sampled_healthy_serials)}")
    print(f"Total entries (rows): {total_entries}")
    print(f"   - Positive (failure==1): {pos_entries}")
    print(f"   - Negative             : {total_entries - pos_entries}")

    return test_ds

In [ ]:
def remove_failure_line(text: str) -> str:
    lines = text.splitlines()
    cleaned_lines = [
        line
        for line in lines
        if not line.strip().startswith("Failure in next 7 days:")
    ]
    return "\n".join(cleaned_lines)

In [ ]:
def process_description(example):
    text = example['description']
    lines = text.split('\n')
    
    cleaned_lines = [
        line for line in lines 
    ]
    
    example['description'] = '\n'.join(cleaned_lines)
    
    if "Failure in next 7 days: Yes" in text:
        example['failure'] = 1
    
    return example

In [ ]:
csv_path = str(MONTHLY_TEXTUAL_DIR)

datasets = load_csv_datasets(
    csv_path,
    pattern=MONTHLY_TEXTUAL_PATTERN,
    seed=42,
    skip_first_n_months=15,
    healthy_sample_divisor=15,
)

rag_datasets = load_historical_datasets(
    csv_path,
    pattern=MONTHLY_TEXTUAL_PATTERN,
    seed=42,
    num_historical_months=15,
    healthy_sample_ratio=0.10,
)

In [ ]:
all_serials = set(datasets.unique("serial_number"))

failure_rows = datasets.filter(lambda x: x["failure"] == 1)
failure_serials = sorted(failure_rows.unique("serial_number")) if len(failure_rows) > 0 else []

healthy_serials = sorted(list(all_serials - set(failure_serials)))

print(f"Number of failure serials: {len(failure_serials)}")
print(f"Number of healthy serials: {len(healthy_serials)}")

print("Failure serials (first 20):", failure_serials[:20])
print("Healthy serials (first 20):", healthy_serials[:20])

In [ ]:
datasets = datasets.map(process_description)
rag_datasets = rag_datasets.map(process_description)

In [ ]:
all_serials = set(datasets.unique("serial_number"))

failure_rows = datasets.filter(lambda x: x["failure"] == 1)
failure_serials = sorted(failure_rows.unique("serial_number")) if len(failure_rows) > 0 else []

healthy_serials = sorted(list(all_serials - set(failure_serials)))

print(f"Number of failure serials: {len(failure_serials)}")
print(f"Number of healthy serials: {len(healthy_serials)}")

## Select Test Subsets

In [ ]:
df = datasets.to_pandas()
df["date"] = pd.to_datetime(df["date"])

failure_per_serial = df.groupby("serial_number")["failure"].sum().reset_index()
failure_per_serial = failure_per_serial.rename(columns={"failure": "failure_count"})

failed_serials = failure_per_serial[
    failure_per_serial["failure_count"] >= 1
]["serial_number"].tolist()

healthy_serials = failure_per_serial[
    failure_per_serial["failure_count"] == 0
]["serial_number"].tolist()

print(f"Number of serials with at least one failure: {len(failed_serials)}")
print(f"Number of fully healthy serials with no failure: {len(healthy_serials)}")

df["description"] = df["description"].fillna("")
df["description_len"] = df["description"].astype(str).apply(len)

healthy_df = df[df["serial_number"].isin(healthy_serials)].copy()

healthy_desc_stats = (
    healthy_df.groupby("serial_number")["description_len"]
    .mean()
    .reset_index()
    .rename(columns={"description_len": "avg_description_len"})
)

healthy_desc_stats = healthy_desc_stats.sort_values("avg_description_len").reset_index(drop=True)

split_idx = len(healthy_desc_stats) // 2
shorter_pool = healthy_desc_stats.iloc[:split_idx]["serial_number"].tolist()
longer_pool = healthy_desc_stats.iloc[split_idx:]["serial_number"].tolist()

print(f"Number of healthy serials in shorter_pool: {len(shorter_pool)}")
print(f"Number of healthy serials in longer_pool: {len(longer_pool)}")

random.seed(42)

n_failed = 30
selected_failed = random.sample(failed_serials, n_failed) if len(failed_serials) >= n_failed else failed_serials

n_healthy_total = 60
n_short = 48
n_long = 12

selected_short = random.sample(shorter_pool, min(n_short, len(shorter_pool)))
selected_long = random.sample(longer_pool, min(n_long, len(longer_pool)))

selected_healthy = selected_short + selected_long

if len(selected_healthy) < n_healthy_total:
    already_selected = set(selected_healthy)
    remaining_healthy = [s for s in healthy_serials if s not in already_selected]
    need_more = n_healthy_total - len(selected_healthy)
    extra = random.sample(remaining_healthy, min(need_more, len(remaining_healthy)))
    selected_healthy.extend(extra)

print(f"Selected failure serials: {len(selected_failed)}")
print(f"Selected healthy serials: {len(selected_healthy)}")
print(f"Selected healthy serials from shorter_pool: {len([s for s in selected_healthy if s in set(shorter_pool)])}")

df_failed = df[df["serial_number"].isin(selected_failed)].copy()
df_failed = df_failed.sort_values(["serial_number", "date"]).reset_index(drop=True)

def sample_fraction_window_for_healthy(
    df_healthy_selected,
    selected_healthy_serials,
    fraction=1/3,
    min_days=1,
    seed=42,
):
    random.seed(seed)
    kept_chunks = []

    for serial in selected_healthy_serials:
        g = df_healthy_selected[df_healthy_selected["serial_number"] == serial].copy()
        g = g.sort_values("date").reset_index(drop=True)

        unique_dates = sorted(g["date"].dropna().unique())
        n_dates = len(unique_dates)

        if n_dates == 0:
            continue

        window_days = max(min_days, math.ceil(n_dates * fraction))

        if window_days >= n_dates:
            kept_chunks.append(g)
            continue

        start_idx = random.randint(0, n_dates - window_days)
        chosen_dates = set(unique_dates[start_idx:start_idx + window_days])

        g_window = g[g["date"].isin(chosen_dates)].copy()
        kept_chunks.append(g_window)

    if not kept_chunks:
        return pd.DataFrame(columns=df_healthy_selected.columns)

    return (
        pd.concat(kept_chunks, axis=0)
        .sort_values(["serial_number", "date"])
        .reset_index(drop=True)
    )

df_healthy_selected = df[df["serial_number"].isin(selected_healthy)].copy()
df_healthy_final = sample_fraction_window_for_healthy(
    df_healthy_selected,
    selected_healthy_serials=selected_healthy,
    fraction=1/3,
    min_days=1,
    seed=42,
)

df_final = pd.concat([df_failed, df_healthy_final], axis=0)
df_final = df_final.sort_values(["serial_number", "date"]).reset_index(drop=True)

print("\n=== Final Dataset Summary ===")
print(f"Final dataset rows: {len(df_final):,}")
print(f"Selected serials: {len(selected_failed)} failure serials + {len(selected_healthy)} healthy serials")
print(f"Total selected serials: {len(selected_failed) + len(selected_healthy)}")

summary = (
    df_final.groupby("serial_number")
    .agg(
        num_rows=("date", "size"),
        num_days=("date", "nunique"),
        failure_count=("failure", "sum"),
    )
    .reset_index()
)

print(summary.head())
print(f"Final number of failure serials: {(summary['failure_count'] > 0).sum()}")
print(f"Final number of healthy serials: {(summary['failure_count'] == 0).sum()}")

final_dataset = Dataset.from_pandas(df_final, preserve_index=False)

In [ ]:
datasets.save_to_disk(str(INFERENCE_DATASETS_DIR))
final_dataset.save_to_disk(str(SAMPLED_INFERENCE_DATASETS_DIR))

## RAG Ingestion Phase

### Preprocessing

In [ ]:
def extract_disk_id_from_description(desc: str) -> str:
    m = re.search(r"(?im)^\s*Disk:\s*(.+?)\s*$", str(desc))
    return m.group(1).strip() if m else ""


def extract_date_from_description(desc: str):
    m = re.search(r"(?im)^\s*Date:\s*(.+?)\s*$", str(desc))
    return pd.to_datetime(m.group(1).strip(), errors="coerce") if m else pd.NaT


def extract_state_signature(desc: str) -> tuple:
    """
    Extract SMART state signature from D2T text.

    Example line:
    smart_187_raw | Reported Uncorrectable Errors | lower the better | current status: high
    smart_7_raw | Seek Error Rate | 7-Day Trend: slightly increasing
    """
    states = []

    for line in str(desc).splitlines():
        line = line.strip()
        if not line.startswith("smart_"):
            continue

        parts = [p.strip() for p in line.split("|")]
        smart_id = parts[0]

        state = None

        # current status
        m_status = re.search(r"current status:\s*([^|]+)", line, flags=re.I)
        if m_status:
            state = "status=" + m_status.group(1).strip().lower()

        # trend
        m_trend = re.search(r"7-day trend:\s*([^|]+)", line, flags=re.I)
        if m_trend:
            state = "trend=" + m_trend.group(1).strip().lower()

        if state is not None:
            states.append((smart_id, state))

    return tuple(sorted(set(states)))


def is_abnormal_signature(signature: tuple) -> bool:
    """
    Decide whether a state signature contains useful abnormal evidence.
    """
    abnormal_keywords = {
        "high",
        "extremely high",
        "low",
        "extremely low",
        "increasing",
        "slightly increasing",
        "decreasing",
        "slightly decreasing",
    }

    for _, state in signature:
        for kw in abnormal_keywords:
            if kw in state:
                return True

    return False


def compress_rag_corpus_by_state_transition(
    rag_datasets,
    anchor_stride: int = 14,
    failure_window: int = 7,
    keep_all_abnormal: bool = False,
):
    df = rag_datasets.to_pandas().copy()

    print("========== RAG State Transition Compression ==========")
    print(f"Original RAG rows: {len(df):,}")

    if "description" not in df.columns:
        raise ValueError("rag_datasets must contain a `description` column.")

    if "failure" not in df.columns:
        raise ValueError("rag_datasets must contain a `failure` column.")

    df["description"] = df["description"].fillna("").astype(str)
    df["failure"] = df["failure"].fillna(0).astype(int)

    if "serial_number" not in df.columns:
        df["serial_number"] = df["description"].apply(extract_disk_id_from_description)

    if "date" not in df.columns:
        df["date"] = df["description"].apply(extract_date_from_description)
    else:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")

    df["_state_signature"] = df["description"].apply(extract_state_signature)
    df["_is_abnormal"] = df["_state_signature"].apply(is_abnormal_signature)

    print("Original label distribution:")
    print(df["failure"].value_counts(dropna=False).to_string())

    kept_groups = []

    for serial, g in df.sort_values(["serial_number", "date"]).groupby("serial_number", sort=False):
        g = g.sort_values("date").reset_index(drop=True)

        keep = pd.Series(False, index=g.index)

        # 1. Always keep first and last observation of each disk
        if len(g) > 0:
            keep.iloc[0] = True
            keep.iloc[-1] = True

        # 2. Always keep failure samples
        failure_mask = g["failure"].astype(int) == 1
        keep |= failure_mask

        # 3. Always keep failure-window context
        failure_indices = list(g.index[failure_mask])
        for idx in failure_indices:
            left = max(0, idx - failure_window)
            right = min(len(g), idx + failure_window + 1)
            keep.iloc[left:right] = True

        # 4. Keep state transition boundaries
        sigs = list(g["_state_signature"])

        for i in range(len(g)):
            prev_sig = sigs[i - 1] if i > 0 else None
            curr_sig = sigs[i]
            next_sig = sigs[i + 1] if i < len(g) - 1 else None

            # state starts here
            if prev_sig is not None and curr_sig != prev_sig:
                keep.iloc[i] = True
                keep.iloc[i - 1] = True

            # state ends here
            if next_sig is not None and curr_sig != next_sig:
                keep.iloc[i] = True
                keep.iloc[i + 1] = True

        # 5. Optionally keep all abnormal states
        if keep_all_abnormal:
            keep |= g["_is_abnormal"]

        # 6. Anchor sampling for long stable regions
        for start in range(0, len(g), anchor_stride):
            keep.iloc[start] = True

        kept_groups.append(g[keep].copy())

    filtered_df = pd.concat(kept_groups, axis=0).reset_index(drop=True)

    print(f"After compression: {len(filtered_df):,}")
    print("Filtered label distribution:")
    print(filtered_df["failure"].value_counts(dropna=False).to_string())
    print(f"Compression ratio: {len(filtered_df) / max(1, len(df)):.4f}")
    print("=====================================================")

    filtered_df = filtered_df.drop(
        columns=["_state_signature", "_is_abnormal"],
        errors="ignore",
    )

    return Dataset.from_pandas(filtered_df, preserve_index=False)

### Ingestions

In [ ]:
MODEL_NAME = 'all-MiniLM-L6-v2'
USE_FP16 = True
BATCH_SIZE = 2048
CHUNK_SIZE = 200_000
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
INDEX_PATH = f'data/{model_serial_id}_RLHF.faiss'

In [ ]:
def parse_disk_report(text):
    lines = [line.strip() for line in text.split('\n') if line.strip()]
    
    basic = {}
    if len(lines) >= 6:
        basic['disk'] = lines[0].split(':', 1)[1].strip() if ':' in lines[0] else ''
        basic['date'] = lines[1].split(':', 1)[1].strip() if ':' in lines[1] else ''
        basic['model'] = lines[2].split(':', 1)[1].strip() if ':' in lines[2] else ''
        basic['production_serial'] = lines[3].split(':', 1)[1].strip() if ':' in lines[3] else ''
        basic['endurance'] = lines[4].split(':', 1)[1].strip() if ':' in lines[4] else ''
        # Failure line ignored for retrieval (it's the label)
    
    smart_attrs = []
    in_smart_section = False
    for line in lines:
        if line.startswith('SMART Features:'):
            in_smart_section = True
            continue
        if in_smart_section and '|' in line:
            parts = [p.strip() for p in line.split('|', 2)]
            if len(parts) == 3:
                attr_id, desc, rel_val_part = parts
                # rel_val_part is like "relative value: low"
                rel_val = rel_val_part.split(':', 1)[1].strip() if ':' in rel_val_part else ''
                smart_attrs.append({
                    'id': attr_id,
                    'desc': desc,
                    'rel_val': rel_val  # e.g., "low", "high", "moderate"
                })
    
    return {
        'basic': basic,
        'smart_attrs': smart_attrs
    }

In [ ]:
all_past_data = list(zip(
    rag_datasets["description"],
    rag_datasets["failure"]
))

all_past_texts = [desc for desc, _ in all_past_data]
metadata = [
    {'description': desc, 'failure': fail}
    for desc, fail in all_past_data
]

print(f"all_texts length: {len(all_past_texts)}")
print(f"metadata length: {len(metadata)}")
print("Sample metadata[0]:", metadata[0])

In [ ]:
parsed_list = []

print("Parsing all descriptions for structured fields...")
for desc in tqdm(all_past_texts):
    parsed = parse_disk_report(desc)
    parsed_list.append(parsed)

print(f"Parsed {len(parsed_list)} examples")

with open(f'data/{model_serial_id}_parsed_list.pkl', 'wb') as f:
    pickle.dump(parsed_list, f)

with open(f'data/{model_serial_id}_metadata.pkl', 'wb') as f:
    pickle.dump(metadata, f)

print("Saved parsed_list and metadata")

In [ ]:
print(f"Loading model '{MODEL_NAME}' on {DEVICE}...")
embed_model = SentenceTransformer(MODEL_NAME, device=DEVICE)

if USE_FP16 and DEVICE.startswith('cuda'):
    print("Using FP16 (half precision) for faster encoding")
    embed_model = embed_model.half()

dimension = embed_model.get_sentence_embedding_dimension()
faiss_index = faiss.IndexFlatIP(dimension)

print(f"Building FAISS index | batch={BATCH_SIZE} | fp16={USE_FP16} | chunks={CHUNK_SIZE:,}")
print(f"Total texts: {len(all_past_texts):,}")

In [ ]:
for start_idx in tqdm(range(0, len(all_past_texts), CHUNK_SIZE), desc="Processing chunks"):
    end_idx = min(start_idx + CHUNK_SIZE, len(all_past_texts))
    chunk_texts = all_past_texts[start_idx:end_idx]

    embeddings = embed_model.encode(
        chunk_texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=False,
        normalize_embeddings=True,
        convert_to_numpy=True,
        device=DEVICE
    )

    if embeddings.dtype != np.float32:
        embeddings = embeddings.astype(np.float32)

    faiss_index.add(embeddings)

    del embeddings
    if DEVICE.startswith('cuda'):
        torch.cuda.empty_cache()

faiss.write_index(faiss_index, INDEX_PATH)
print(f"FAISS index saved to: {INDEX_PATH}")
print(f"Total vectors indexed: {faiss_index.ntotal:,}")

## Inference

In [ ]:
datasets = load_from_disk(str(INFERENCE_DATASETS_DIR))
overall_test_datasets = load_from_disk(str(SAMPLED_INFERENCE_DATASETS_DIR))
df_smart_des = pd.read_csv(SMART_ATTRIBUTES_PATH)

print("Sorting entire test dataset by date for global time-series inference...")
df_test = overall_test_datasets.to_pandas()
df_test['date'] = pd.to_datetime(df_test['date'])
df_test = df_test.sort_values(['serial_number', 'date']).reset_index(drop=True)

print(f"First date: {df_test['date'].min().date()}")
print(f"Last date:  {df_test['date'].max().date()}")
print(f"Total rows: {len(df_test):,}")

In [ ]:
MODEL_NAME = 'all-MiniLM-L6-v2'
USE_FP16 = True
BATCH_SIZE = 2048
CHUNK_SIZE = 200_000
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
INDEX_PATH = f'data/{model_serial_id}_RLHF.faiss'

index = faiss.read_index(f'data/{model_serial_id}_RLHF.faiss')
print("FAISS index loaded")

with open(f'data/{model_serial_id}_metadata.pkl', 'rb') as f:
    metadata = pickle.load(f)
print("Metadata loaded successfully")

with open(f'data/{model_serial_id}_parsed_list.pkl', 'rb') as f:
    parsed_list = pickle.load(f)
print(f"Loaded {len(parsed_list)} parsed examples")

In [ ]:
serial_to_indices = defaultdict(list)
model_to_indices = defaultdict(list)
smart_id_value_to_indices = defaultdict(lambda: defaultdict(list))
all_smart_ids_per_index = []

for idx, parsed in enumerate(parsed_list):
    basic = parsed['basic']
    
    serial = basic.get('production_serial')
    if serial:
        serial_to_indices[serial].append(idx)
    
    disk_model = basic.get('model', '').lower()
    if disk_model:
        model_to_indices[disk_model].append(idx)
    
    smart_set = set()
    for attr in parsed['smart_attrs']:
        attr_id = attr['id']
        rel_val = attr['rel_val']
        smart_id_value_to_indices[attr_id][rel_val].append(idx)
        smart_set.add(attr_id)
    all_smart_ids_per_index.append(smart_set)

In [ ]:
model_path = FINAL_ADAPTER_DIR
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_storage=torch.uint8
)

base_model_id = "microsoft/phi-4"
model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

model = PeftModel.from_pretrained(model, model_path)

model.eval()

In [ ]:
SYSTEM_PROMPT = (
    "You are an expert in hard drive failure prediction. "
    "Analyze the provided disk report and respond with the word either 'Yes' or 'No'. "
    "Strictly do not include any explanations or additional text, just include Yes or No!"
)

USER_INSTRUCTION_TEMPLATE = (
    "Please analyze the disk report: \n{description}\n\n"
    "Only some of the collected SMART attributes are shown in the report. \n\n"
    "A strong machine learning model's prediction on this disk is: {prediction_base}. \n\n"
    "You may also reference some of the related cases and their actual health status: \n"
    "{related_cases}\n"
    "Consider the above evidences, will this hard drive likely fail within the next 7 days and therefore warrant replacement? Answer either 'Yes' or 'No'. "
)

def build_fallback_prompt(description, prediction_base, related_cases):
    if ABLATION_OPTION == 0:
        user_content = USER_INSTRUCTION_TEMPLATE.format(description=description, prediction_base=prediction_base, related_cases=related_cases)
    elif ABLATION_OPTION == 1:
        USER_INSTRUCTION_TEMPLATE = (
            "Please analyze the disk report: \n{description}\n\n"
            "Only some of the collected SMART attributes are shown in the report. \n\n"
            "A strong machine learning model's prediction on this disk is: {prediction_base}. \n\n"
            "Consider the above evidences, will this hard drive likely fail within the next 7 days and therefore warrant replacement? Answer either 'Yes' or 'No'. "
        )
        user_content = USER_INSTRUCTION_TEMPLATE.format(description=description, prediction_base=prediction_base)
    elif ABLATION_OPTION == 2:
        USER_INSTRUCTION_TEMPLATE = (
            "Please analyze the disk report: \n{description}\n\n"
            "Only some of the collected SMART attributes are shown in the report. \n\n"
            "You may also reference some of the related cases and their actual health status: \n"
            "{related_cases}\n"
            "Consider the above evidences, will this hard drive likely fail within the next 7 days and therefore warrant replacement? Answer either 'Yes' or 'No'. "
        )
        user_content = USER_INSTRUCTION_TEMPLATE.format(description=description, related_cases=related_cases)
    else:
        USER_INSTRUCTION_TEMPLATE = (
            "Please analyze the disk report: \n{description}\n\n"
            "Only some of the collected SMART attributes are shown in the report. \n\n"
            "Consider the above evidences, will this hard drive likely fail within the next 7 days and therefore warrant replacement? Answer either 'Yes' or 'No'. "
        )
        user_content = USER_INSTRUCTION_TEMPLATE.format(description=description)
    return (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{user_content}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

In [ ]:
def parse_disk_report(text):
    lines = [line.strip() for line in text.split('\n') if line.strip()]
    
    basic = {}
    if len(lines) >= 6:
        basic['disk'] = lines[0].split(':', 1)[1].strip() if ':' in lines[0] else ''
        basic['date'] = lines[1].split(':', 1)[1].strip() if ':' in lines[1] else ''
        basic['model'] = lines[2].split(':', 1)[1].strip() if ':' in lines[2] else ''
        basic['production_serial'] = lines[3].split(':', 1)[1].strip() if ':' in lines[3] else ''
        basic['endurance'] = lines[4].split(':', 1)[1].strip() if ':' in lines[4] else ''
        # Failure line ignored for retrieval (it's the label)
    
    smart_attrs = []
    in_smart_section = False
    for line in lines:
        if line.startswith('SMART Features:'):
            in_smart_section = True
            continue
        if in_smart_section and '|' in line:
            parts = [p.strip() for p in line.split('|', 2)]
            if len(parts) == 3:
                attr_id, desc, rel_val_part = parts
                # rel_val_part is like "relative value: low"
                rel_val = rel_val_part.split(':', 1)[1].strip() if ':' in rel_val_part else ''
                smart_attrs.append({
                    'id': attr_id,
                    'desc': desc,
                    'rel_val': rel_val  # e.g., "low", "high", "moderate"
                })
    
    return {
        'basic': basic,
        'smart_attrs': smart_attrs
    }

In [ ]:
def clean_old_tags(text: str) -> str:
    """Remove all chat template tags and any assistant responses from old examples."""
    text = re.sub(r'<\|.*?\|>', '', text)
    text = re.sub(r'>[01]\s*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'\n\s*\n', '\n\n', text)
    return text.strip()

def extract_smart_attributes(text: str) -> Set[str]:
    """Extract mentioned SMART attribute names like smart_9_raw, smart_197_raw, etc."""
    return set(re.findall(r'\b(smart_\d+_raw)\b', text.lower()))

In [ ]:
def create_inference_prompt(
    description: str,
    retrieved_context: str,
    prediction_base: str,
    df_smart_des
) -> str:
    clean_description = clean_old_tags(description)
    clean_context = clean_old_tags(retrieved_context)

    formatted_examples = ""
    
    if ABLATION_OPTION == 0 or ABLATION_OPTION == 2:
        if clean_context:
            formatted_examples = clean_context
        else:
            formatted_examples = "No past examples available."

    all_text = clean_description + "\n" + clean_context
    
    prompt = build_fallback_prompt(clean_description, prediction_base, formatted_examples)
    return prompt

In [ ]:
def generate_response(model, tokenizer, prompt, max_new_tokens=100, temperature=0.0):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(model.device)
    
    lengths = inputs["length"] if "length" in inputs else [len(ids) for ids in inputs["input_ids"]]
    # print("length of input is:", lengths)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

    response = re.sub(r'<think>.*?</think>', '', response, flags=re.DOTALL)
    response = re.sub(r'<\|[^>]+?\>', '', response)
    response = re.sub(r'[^a-zA-Z\s]', '', response).strip().lower()

    return response

In [ ]:
def sanitize_filename(name: str) -> str:
    return (
        name.replace("/", "_")
            .replace(" ", "_")
            .replace("(", "")
            .replace(")", "")
    )

In [ ]:
SCRIPT_DIR = Path(__file__).resolve().parent

RF_BASELINE_ROOT = (
    SCRIPT_DIR / "../rf_ml_baseline"
).resolve()

RF_RESULT_PATH = (
    RF_BASELINE_ROOT
    / "outputs"
    / "results"
    / f"{sanitize_filename(model_serial_id)}_local_inference_results.csv"
)

CALIBRATOR_PATH = (
    RF_BASELINE_ROOT
    / "data"
    / "features"
    / "rf_calibrator.joblib"
)

TRAIN_CALIBRATOR_PATH = (
    RF_BASELINE_ROOT
    / "train_calibrator.py"
)


if not RF_RESULT_PATH.exists():
    raise FileNotFoundError(
        f"RF result file not found: {RF_RESULT_PATH}"
    )

if not CALIBRATOR_PATH.exists():
    raise FileNotFoundError(
        f"RF calibrator not found: {CALIBRATOR_PATH}"
    )

if not TRAIN_CALIBRATOR_PATH.exists():
    raise FileNotFoundError(
        f"train_calibrator.py not found: {TRAIN_CALIBRATOR_PATH}"
    )


# Load build_features() directly from train_calibrator.py.
spec = importlib.util.spec_from_file_location(
    "rf_train_calibrator",
    TRAIN_CALIBRATOR_PATH,
)

if spec is None or spec.loader is None:
    raise ImportError(
        f"Unable to load module from: {TRAIN_CALIBRATOR_PATH}"
    )

train_calibrator_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(train_calibrator_module)

build_features = train_calibrator_module.build_features


# Load RF results and the trained calibrator.
rf_results = pd.read_csv(RF_RESULT_PATH)
calibrator = joblib.load(CALIBRATOR_PATH)

pipeline = calibrator["pipeline"]
threshold = float(calibrator["threshold"])
feature_columns = calibrator["feature_columns"]


# Build temporal features and generate calibrated predictions.
rf_results_new = build_features(rf_results)

rf_results_new["y_new_prob"] = pipeline.predict_proba(
    rf_results_new[feature_columns]
)[:, 1]

rf_results_new["y_new"] = (
    rf_results_new["y_new_prob"] >= threshold
).astype(int)


# Build calibrated RF evidence for LexDFP.
_ml_df = rf_results_new.copy()
_ml_df["date"] = pd.to_datetime(_ml_df["date"]).dt.date

ml_lookup = {
    (row.serial_number, row.date): (
        "Yes" if int(row.y_new) == 1 else "No"
    )
    for row in _ml_df.itertuples()
}

In [ ]:
_WORKER_MODEL_NAME = "all-MiniLM-L6-v2"
_WORKER_INDEX_PATH = f"data/{model_serial_id}_RLHF.faiss"

_worker_embed_model = None
_worker_faiss_index = None
_worker_dimension = None
_worker_metadata = None
_worker_parsed_list = None
_worker_serial_to_indices = None
_worker_model_to_indices = None
_worker_smart_id_value_to_indices = None
_worker_all_smart_ids_per_index = None
_worker_df_smart_des = None
_worker_ml_lookup = None

def _init_prompt_worker(
    metadata_,
    parsed_list_,
    serial_to_indices_,
    model_to_indices_,
    smart_id_value_to_indices_,
    all_smart_ids_per_index_,
    df_smart_des_,
    ml_lookup_,
):
    global _worker_embed_model
    global _worker_faiss_index
    global _worker_dimension
    global _worker_metadata
    global _worker_parsed_list
    global _worker_serial_to_indices
    global _worker_model_to_indices
    global _worker_smart_id_value_to_indices
    global _worker_all_smart_ids_per_index
    global _worker_df_smart_des
    global _worker_ml_lookup

    os.environ["OMP_NUM_THREADS"] = "1"
    os.environ["MKL_NUM_THREADS"] = "1"

    try:
        torch.set_num_threads(1)
        torch.set_num_interop_threads(1)
    except Exception:
        pass

    _worker_embed_model = SentenceTransformer(_WORKER_MODEL_NAME, device="cpu")
    _worker_faiss_index = faiss.read_index(_WORKER_INDEX_PATH)
    _worker_dimension = _worker_embed_model.get_sentence_embedding_dimension()

    _worker_metadata = metadata_
    _worker_parsed_list = parsed_list_
    _worker_serial_to_indices = serial_to_indices_
    _worker_model_to_indices = model_to_indices_
    _worker_smart_id_value_to_indices = smart_id_value_to_indices_
    _worker_all_smart_ids_per_index = all_smart_ids_per_index_
    _worker_df_smart_des = df_smart_des_
    _worker_ml_lookup = ml_lookup_

In [ ]:
def _remove_failure_line(text: str) -> str:
    lines = text.splitlines()
    cleaned_lines = [
        line
        for line in lines
        if not line.strip().startswith("Failure in next 7 days:")
    ]
    return "\n".join(cleaned_lines)

def _rewrite_failure_line(text: str, label: bool) -> str:
    lines = text.splitlines()
    new_lines = []

    for line in lines:
        stripped = line.strip()

        if re.match(r"Failure in next 7 days:\s*Yes\.?", stripped, re.IGNORECASE) or (re.match(r"Failure in next 7 days:\s*No\.?", stripped, re.IGNORECASE) and label):
            new_lines.append(
                "The disk is highly expected to experience a failure in the next 7 days."
            )

        elif re.match(r"Failure in next 7 days:\s*No\.?", stripped, re.IGNORECASE) and not label:
            new_lines.append(
                "The disk is not likely to experience a failure in the next 7 days."
            )

        else:
            new_lines.append(line)

    return "\n".join(new_lines)

def _structured_match_score_local(query_parsed, cand_parsed):
    score = 0.0

    q_basic = query_parsed['basic']
    c_basic = cand_parsed['basic']

    if q_basic.get('production_serial') == c_basic.get('production_serial'):
        score += 50.0

    q_model = q_basic.get('model', '').lower()
    c_model = c_basic.get('model', '').lower()
    if q_model and c_model and (q_model in c_model or c_model in q_model):
        score += 25.0

    if q_basic.get('endurance') == c_basic.get('endurance'):
        score += 8.0

    q_attr_dict = {a['id']: a['rel_val'] for a in query_parsed['smart_attrs']}
    c_attr_dict = {a['id']: a['rel_val'] for a in cand_parsed['smart_attrs']}

    shared_ids = set(q_attr_dict.keys()) & set(c_attr_dict.keys())

    for attr_id in shared_ids:
        q_val = q_attr_dict[attr_id]
        c_val = c_attr_dict[attr_id]

        if q_val == c_val:
            score += 40.0
        else:
            score -= 80.0

    score += len(shared_ids) * 2.0
    return score

def _retrieve_examples_cpu_local(query_text, k=2, struct_boost=2.0, max_struct_candidates=200, fallback_search_k_multiplier=10):
    query_parsed = parse_disk_report(query_text)
    q_basic = query_parsed['basic']

    candidate_set = set()

    q_serial = q_basic.get('production_serial')
    if q_serial:
        candidate_set.update(_worker_serial_to_indices[q_serial])

    q_model = q_basic.get('model', '').lower()
    if q_model and q_model in _worker_model_to_indices:
        candidate_set.update(_worker_model_to_indices[q_model])

    has_smart = len(query_parsed['smart_attrs']) > 0
    if has_smart:
        smart_matches = set()
        for attr in query_parsed['smart_attrs']:
            attr_id = attr['id']
            q_val = attr['rel_val']
            matches = _worker_smart_id_value_to_indices[attr_id].get(q_val, [])
            smart_matches.update(matches)
        candidate_set.update(smart_matches)

    if len(candidate_set) < 10 and has_smart:
        q_smart_ids = {a['id'] for a in query_parsed['smart_attrs']}
        for idx, cand_smart_ids in enumerate(_worker_all_smart_ids_per_index):
            if q_smart_ids & cand_smart_ids:
                candidate_set.add(idx)

    candidate_indices = list(candidate_set)

    candidate_scores = []
    for idx in candidate_indices:
        score = _structured_match_score_local(query_parsed, _worker_parsed_list[idx])
        if score > 0:
            candidate_scores.append((idx, score))

    candidate_scores.sort(key=lambda x: x[1], reverse=True)
    candidate_indices = [idx for idx, _ in candidate_scores[:max_struct_candidates]]

    query_emb = _worker_embed_model.encode(
        [query_text],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    ).astype("float32")

    final_candidates = []

    if not candidate_indices:
        search_k = min(k * fallback_search_k_multiplier, _worker_faiss_index.ntotal)
        D, I = _worker_faiss_index.search(query_emb, search_k)
        for dist, idx in zip(D[0], I[0]):
            if idx == -1:
                continue
            final_candidates.append((float(dist), int(idx)))
    else:
        subset_size = len(candidate_indices)
        subset_embeddings = np.empty((subset_size, _worker_dimension), dtype="float32")

        for pos, idx in enumerate(candidate_indices):
            subset_embeddings[pos] = _worker_faiss_index.reconstruct(int(idx))

        dense_scores = np.dot(subset_embeddings, query_emb.T).flatten()
        struct_score_map = dict(candidate_scores)

        for pos, dense_score in enumerate(dense_scores):
            orig_idx = candidate_indices[pos]
            struct_score = struct_score_map.get(orig_idx, 0.0)
            combined_score = float(dense_score) + struct_score * struct_boost
            final_candidates.append((combined_score, orig_idx))

    if not final_candidates:
        return "No similar past examples found."

    final_candidates.sort(key=lambda x: x[0], reverse=True)
    pool = final_candidates[:50]

    failing = sorted(
        [c for c in pool if _worker_metadata[c[1]]['failure'] == 1],
        key=lambda x: x[0],
        reverse=True
    )
    non_failing = sorted(
        [c for c in pool if _worker_metadata[c[1]]['failure'] == 0],
        key=lambda x: x[0],
        reverse=True
    )

    selected_indices = []
    seen = set()

    if non_failing:
        idx_nf = non_failing[0][1]
        selected_indices.append(idx_nf)
        seen.add(idx_nf)

    if failing:
        idx_f = failing[0][1]
        selected_indices.append(idx_f)
        seen.add(idx_f)

    for score, idx in pool:
        if idx not in seen:
            selected_indices.append(idx)
            seen.add(idx)
            if len(selected_indices) >= k:
                break

    if len(selected_indices) < k:
        for score, idx in final_candidates:
            if idx not in seen:
                selected_indices.append(idx)
                seen.add(idx)
                if len(selected_indices) >= k:
                    break

    selected_indices = selected_indices[:k]

    retrieved_examples = []
    for idx_count, idx in enumerate(selected_indices):
        ex = _worker_metadata[idx]
        clean_report = ex['description']
        label = ex['failure']
        decision = "Yes" if label == 1 else "No"

        example_block = f"""
Past Drive Report {idx_count}:
{clean_report}

Failure Status {idx_count}: {decision}

"""
        retrieved_examples.append(example_block)

    return "\n".join(retrieved_examples).strip() if retrieved_examples else "No similar past examples found."

In [ ]:
def _build_single_prompt(task):
    idx, row = task

    label = row['failure']
    text = _remove_failure_line(row["description"])
    row_date = pd.Timestamp(row["date"]).date()
    row_serial = row["serial_number"]

    if ABLATION_OPTION == 1 or ABLATION_OPTION == 3:
        retrieved_context = ""
    else:
        retrieved_context = _retrieve_examples_cpu_local(text, k=2)

    ml_response = _worker_ml_lookup.get((row_serial, row_date), "No")

    prompt = create_inference_prompt(
        description=text,
        retrieved_context=retrieved_context,
        prediction_base=ml_response,
        df_smart_des=_worker_df_smart_des
    )

    return idx, prompt

In [ ]:
tasks = [(idx, row) for idx, row in df_test.iterrows()]
prompts = [None] * len(df_test)

num_workers = max(1, min((os.cpu_count() or 8) - 2, 10))
print(f"Building prompts with {num_workers} CPU workers...")

with ProcessPoolExecutor(
    max_workers=num_workers,
    initializer=_init_prompt_worker,
    initargs=(
        metadata,
        parsed_list,
        serial_to_indices,
        model_to_indices,
        smart_id_value_to_indices,
        all_smart_ids_per_index,
        df_smart_des,
        ml_lookup,
    )
) as executor:
    futures = [executor.submit(_build_single_prompt, task) for task in tasks]

    for future in tqdm(as_completed(futures), total=len(futures), desc="Prompt building"):
        idx, prompt = future.result()
        prompts[idx] = prompt

df_test["prompt"] = prompts

print("Prompt building finished.")
print(df_test[["serial_number", "date", "failure", "prompt"]].head(3))


# Alternative output paths for ablation runs:
if ABLATION_OPTION == 0:
    save_path = DF_TEST_PICKLE_PATH
elif ABLATION_OPTION == 1:
    save_path = INFERENCE_DATA_DIR / "df_test_no_rag.pkl"
elif ABLATION_OPTION == 2:
    save_path = INFERENCE_DATA_DIR / "df_test_no_ml.pkl"
else:
    save_path = INFERENCE_DATA_DIR / "df_test_no_rag_no_ml.pkl"

df_test.to_pickle(save_path)
print(f"Saved to: {save_path}")

# with open(save_path, 'rb') as file:
#     df_test = pickle.load(file)

In [ ]:
print("Sorting test dataframe for per-serial chronological inference...")
df_test = df_test.sort_values(['serial_number', 'date']).reset_index(drop=True)

# Optional: keep original row ids so nothing gets misaligned
df_test["_row_id"] = range(len(df_test))

def generate_batch_responses(model, tokenizer, prompts, max_new_tokens=20):
    """
    Batched deterministic generation for yes/no prompts.
    Output order matches input prompt order exactly.
    """
    tokenizer.padding_side = "left"

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        truncation=True,
        max_length=1024,
        padding=True
    )

    # Move tensors onto the model device
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    input_len = inputs["input_ids"].shape[1]
    decoded = tokenizer.batch_decode(
        outputs[:, input_len:],
        skip_special_tokens=True
    )

    cleaned = []
    for response in decoded:
        response = re.sub(r'<think>.*?</think>', '', response, flags=re.DOTALL)
        response = re.sub(r'<\|[^>]+?\>', '', response)
        response = re.sub(r'[^a-zA-Z\s0-9]', '', response).strip().lower()
        cleaned.append(response)

    return cleaned


def parse_binary_prediction(clean_response):
    """
    Convert model text output into 0/1 prediction.
    """
    if "no" in clean_response or clean_response == "0" or clean_response.startswith("0"):
        return 0
    if "yes" in clean_response or clean_response == "1" or clean_response.startswith("1"):
        return 1
    return None

In [ ]:
batch_size = 24

responses = [None] * len(df_test)
recommendations = [None] * len(df_test)
labels = [None] * len(df_test)
bases = [None] * len(df_test)
dates = [None] * len(df_test)
serial_ids = [None] * len(df_test)

print("Starting per-serial batched inference...")
for serial, disk_df in tqdm(
    df_test.groupby('serial_number', sort=False),
    total=df_test['serial_number'].nunique()
):
    # Preserve chronological order within each serial
    disk_df = disk_df.sort_values('date').reset_index(drop=False)   # keep original index in sorted df_test

    prompts = disk_df["prompt"].tolist()

    # Optional: keep ML base if you stored it during prompt building
    if "ml_base" in disk_df.columns:
        ml_bases_this_serial = disk_df["ml_base"].tolist()
    else:
        ml_bases_this_serial = [None] * len(disk_df)

    true_labels_this_serial = disk_df["failure"].tolist()
    dates_this_serial = disk_df["date"].tolist()
    row_ids_this_serial = disk_df["index"].tolist()  # original row positions in sorted df_test

    for start in range(0, len(disk_df), batch_size):
        end = min(start + batch_size, len(disk_df))

        batch_prompts = prompts[start:end]
        batch_labels = true_labels_this_serial[start:end]
        batch_dates = dates_this_serial[start:end]
        batch_row_ids = row_ids_this_serial[start:end]
        batch_bases = ml_bases_this_serial[start:end]

        batch_responses = generate_batch_responses(
            model,
            tokenizer,
            batch_prompts,
            max_new_tokens=20
        )

        # Retry only ambiguous outputs, still preserving order
        for j, clean_response in enumerate(batch_responses):
            pred = parse_binary_prediction(clean_response)

            if pred is None:
                retry_prompt = batch_prompts[j]

                retry_response = generate_batch_responses(
                    model,
                    tokenizer,
                    [retry_prompt],
                    max_new_tokens=10
                )[0]

                clean_response = retry_response
                pred = parse_binary_prediction(clean_response)

                if pred is None:
                    pred = 0

            row_id = batch_row_ids[j]

            responses[row_id] = clean_response
            recommendations[row_id] = pred
            labels[row_id] = batch_labels[j]
            bases[row_id] = batch_bases[j]
            dates[row_id] = batch_dates[j]
            serial_ids[row_id] = serial

print("Inference finished.")

In [ ]:
temp_bases = []
for serial, disk_df in tqdm(df_test.groupby('serial_number'), 
                           total=df_test['serial_number'].nunique()):
    disk_df = disk_df.sort_values('date').reset_index(drop=True)
    
    for _, row in disk_df.iterrows():
        prompt = row['prompt']
        
        match = re.search(
            r'A strong machine learning model\'s prediction on this disk is:\s*(Yes|No)\b',
            # r'The machine learning method predict the disk to have high failure:\s*(Yes|No)\b',
            prompt,
            re.IGNORECASE
        )
        
        if match:
            value_str = match.group(1).strip().lower()
            if value_str == 'no':
                temp_bases.append(0)
            elif value_str == 'yes':
                temp_bases.append(1)
            else:
                temp_bases.append(None)  # or -1, np.nan, etc.
                # Optional: print warning
                # print(f"Unexpected ML value for {serial} on {row['date']}: {value_str}")
        else:
            temp_bases.append(None)

In [ ]:
results_df = pd.DataFrame({
    "date": dates,
    "serial_number": serial_ids,
    "true_label": labels,
    "ml_base": temp_bases,
    "llm_response": responses,
    "prediction": recommendations,
})

results_df = results_df.sort_values(by=['serial_number', 'date'])
group_sum = results_df.groupby('serial_number')['true_label'].transform('sum')
mask = (group_sum != 0)
last_rows = results_df[mask].groupby('serial_number').tail(1).index
results_df.loc[last_rows, 'true_label'] = 1

results_df["correct"] = (results_df["true_label"] == results_df["prediction"]).astype(int)

if ABLATION_OPTION == 1:
    RESULT_PATH = RESULTS_DIR / f"{model_serial_id}_no_rag.csv"
elif ABLATION_OPTION == 2:
    RESULT_PATH = RESULTS_DIR / f"{model_serial_id}_no_ml.csv"
else:
    RESULT_PATH = RESULTS_DIR / f"{model_serial_id}_no_rag_no_ml.csv"

results_df.to_csv(RESULT_PATH, index=False)

tp = ((results_df["true_label"] == 1) & (results_df["prediction"] == 1)).sum()
tn = ((results_df["true_label"] == 0) & (results_df["prediction"] == 0)).sum()
fp = ((results_df["true_label"] == 0) & (results_df["prediction"] == 1)).sum()
fn = ((results_df["true_label"] == 1) & (results_df["prediction"] == 0)).sum()

total = tp + tn + fp + fn
accuracy = (tp + tn) / len(results_df)
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
fpr = fp / (fp + tn)

print(f"TP: {tp}, TN: {tn}, FP: {fp}, FN: {fn}, TOTAL: {total}")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1       : {f1:.4f}")
print(f"FPR       : {fpr:.4f}")

event_tp_count = results_df.groupby('serial_number').apply(
    lambda g: ((g['true_label'] == 1) & (g['prediction'] == 1)).any()
).sum()
total_failure_count = results_df[results_df['true_label'] == 1]['serial_number'].nunique()

print(f"Event-Level Recall:  {event_tp_count/total_failure_count}")